In [1]:
# # Evaluacion 2 - Programación Científica 2026-1
# # ID: 841

# %% [markdown]
# ## 1. Setup e importaciones

# %%
# Instalación de las librerías necesarias para el ejercicio
!pip install jax jaxlib plotly -q

import sys
import warnings
warnings.filterwarnings('ignore')  # Suprime mensajes de advertencialectura final mas limpia)

import numpy as np  # Operaciones numéricas estándar
import jax  # Framework para diferenciación automática y GPU(optimizacion de machine learning)
import jax.numpy as jnp  # Versión de numpy con soporte JAX
from jax import random, grad, jit  # Herramientas específicas de JAX
import plotly.graph_objects as go  # Gráficas interactivas
from plotly.subplots import make_subplots  # Subgráficas en Plotly,mas de una grafica en una figura
import plotly.io as pio
pio.renderers.default = 'colab'  # Configura para renderizado en colab

print("Configuracion completada con Plotly")

# %% [markdown]
# ## 2. Parámetros de la red

# %%
N = 256  # Longitud de la señal (número de muestras)
K = 25  # Número de frecuencias (usaremos senos y cosenos)
LR = 0.05  # Learning rate (tasa de aprendizaje)
EPOCHS = 3000  # Número de iteraciones de entrenamiento

print(f"Parametros: N={N}, K={K}, LR={LR}, EPOCHS={EPOCHS}")

# %% [markdown]
# ## 3. Cargamos datos

# %%
import os

# Verifica si el archivo de datos existe, si no permite subirlo
if not os.path.exists('../SolucionReferencia_0000/datos_0000.npz'):
    print("Archivo 'datos_0000.npz' no encontrado.")
    from google.colab import files
    uploaded = files.upload()

# Carga los datos del archivo .npz
data = np.load('../SolucionReferencia_0000/datos_0000.npz')
signals = data['X']  # Matriz de 30 señales × 256 muestras
regimenes = data['y']  # Etiquetas: 0=sin dispersión, 1=intermedia, 2=fuerte

print(f"Señales cargadas: {signals.shape}")
print(f"Régimenes: {regimenes}")

# Cuenta cuántas señales hay por régimen
for r in [0, 1, 2]:
    print(f"  Régimen {r}: {np.sum(regimenes == r)} señales")

# %% [markdown]
# ## 4. Implementacion de la red neuronal

# %%
def fourier_matrix(N, K):
    """
    Crea la matriz de Fourier con senos y cosenos para k=1,...,K.
    una matriz de tamaño (2K × N)
    Cada fila es una base: sin(2π·1·t), cos(2π·1·t), ..., sin(2π·K·t), cos(2π·K·t)
    """
    t = jnp.linspace(0, 1, N)  # Vector de tiempo de 0 a 1 con N puntos
    W1 = []
    for k in range(1, K+1):
        W1.append(jnp.sin(2 * jnp.pi * k * t))  # Seno de frecuencia k
        W1.append(jnp.cos(2 * jnp.pi * k * t))  # Coseno de frecuencia k
    return jnp.array(W1)

def init_network(key, N, K, init_type='fourier'):
    """
    Inicializa los pesos de la red.
    - 'fourier': W1 es la matriz de Fourier fija
    - 'random': W1 es aleatoria (distribución normal escalada)
    W2 siempre es aleatoria (capa de salida)
    """
    if init_type == 'fourier':
        W1 = fourier_matrix(N, K)  # Usa la base de Fourier predefinida
    else:
        key, subkey = random.split(key)
        W1 = random.normal(subkey, (2*K, N)) * 0.01  # Pesos aleatorios pequeños

    key, subkey = random.split(key)
    W2 = random.normal(subkey, (N, 2*K)) * 0.01  # Capa de salida aleatoria
    return W1, W2

def forward(W1, W2, x):
    """
    Propagación hacia adelante (forward pass).
    Arquitectura: x → [W1] → h → tanh → [W2] → x_hat
    - h = tanh(W1 @ x): capa oculta con 30 neuronas
    - x_hat = W2 @ h: reconstrucción de la señal (256 muestras)
    """
    h = jnp.tanh(W1 @ x)  # Activación no lineal
    x_hat = W2 @ h  # Reconstrucción
    return x_hat

def loss_fn(W1, W2, x):
    """
    Función de pérdida: Error Cuadrático Medio (MSE)
    L = (1/N) * Σ(ŷ_i - y_i)²
    Mide qué tan similar es el dato de entraa al de salida
    """
    x_hat = forward(W1, W2, x)
    return jnp.mean((x_hat - x) ** 2)

@jit  # Compilación JIT para acelerar el entrenamiento
# (se usa cuando se llama muchas veces una funcion)
def update(W1, W2, x, lr):
    """
    Actualiza los pesos usando Gradiente Descendiente.
    Calcula gradientes de la pérdida con respecto a W1 y W2,
    luego actualiza: W_new = W - lr * gradiente
    para disminuir el error de estos mismos
    """
    grads = grad(loss_fn, argnums=(0, 1))(W1, W2, x)  # Gradientes
    return W1 - lr * grads[0], W2 - lr * grads[1]  # Actualización

def train_network(x, init_type='fourier', epochs=3000, lr=0.05, verbose=True):
    """
    Entrena la red para una señal específica,
    con un numero especifico de epocas.
    - init_type: 'fourier' o 'random'
    - epochs: número de iteraciones
    - lr: learning rate
    - verbose: muestra progreso cada 200 épocas
    Retorna los pesos entrenados y el historial de pérdidas
    """
    key = random.PRNGKey(42)  # Semilla fija para reproducibilidad
    W1, W2 = init_network(key, N, K, init_type)  # Inicialización

    loss_history = []
    for epoch in range(epochs):
        W1, W2 = update(W1, W2, x, lr)  # Actualiza pesos
        loss = loss_fn(W1, W2, x)  # Calcula pérdida actual
        loss_history.append(float(loss))

        if verbose and epoch % 200 == 0:  # Muestra progreso cada 200 épocas
            print(f"  Epoch {epoch:4d}: Loss = {loss:.6f}")

    return W1, W2, loss_history

print("La red neuronal se a implementado")

# %% [markdown]
# ## 5. Verificar transformada de Fourier (con PLOTLY)

# %%
def verificar_fourier(signals, idx=0):
    """
    Verifica que la capa W1 calcula efectivamente la Transformada de Fourier.
    Muestra la señal original y sus coeficientes de Fourier.
    """
    print(f"/Verificando Fourier para señal {idx}...")
    x = signals[idx]  # Señal que se va a analizar
    W1_f = fourier_matrix(N, K)  # Matriz de Fourier
    coefs = np.array(W1_f @ x)  # Coeficientes = W1 @ x (proyección sobre bases)

    # Crear figura con 2 subplots: señal original y coeficientes
    fig = make_subplots(rows=2, cols=1,
                        subplot_titles=(f'Señal original - Régimen {regimenes[idx]}',
                                       f'Coeficientes de Fourier (k=1,...,{K})'))

    # Gráfica 1: Señal original
    fig.add_trace(go.Scatter(y=x, mode='lines', name='Señal', line=dict(width=2)), row=1, col=1)
    fig.update_xaxes(title_text='Tiempo (muestras)', row=1, col=1)
    fig.update_yaxes(title_text='Amplitud', row=1, col=1)

    # Gráfica 2: Coeficientes de Fourier (senos y cosenos)
    fig.add_trace(go.Bar(y=coefs, name='Coeficientes', marker_color='#2E86AB'), row=2, col=1)
    fig.update_xaxes(title_text='Coeficiente (par = seno, impar = coseno)', row=2, col=1)
    fig.update_yaxes(title_text='Magnitud', row=2, col=1)

    fig.update_layout(height=800, width=1000, showlegend=False)
    fig.show()

    return coefs

print("/Verificando Fourier...")
coefs = verificar_fourier(signals, 0)
print(f"Primeros 5 coeficientes: {coefs[:5]}")

# %% [markdown]
# ## 6. Entrenamiento y experimentos

# %%
def entrenar_y_analizar(signals, regimenes, idx_signal, mostrar_reconstruccion=True):
    """
    Entrena la red con ambas inicializaciones (Fourier y aleatoria) para una señal.
    Genera gráficas comparativas de:
    1. Evolución de la pérdida durante el entrenamiento
    2. Pérdida final (comparación en barras)
    3. Reconstrucción de la señal
    4. Error de reconstrucción (permite saber cual iniciazion funciona mejor)
    (estos dos ultimos sirven para saber que tambien esta aprendiendo la red)
    """
    x = signals[idx_signal]
    regime = regimenes[idx_signal]

    print(f"\n{'='*60}")
    print(f"Señal {idx_signal}, Régimen {regime}")
    print(f"{'='*60}")

    # Entrenamiento con inicialización Fourier
    print("[Inicialización Fourier]")
    print(f"  Entrenando {EPOCHS} épocas...")
    W1_f, W2_f, loss_f = train_network(x, 'fourier', EPOCHS, LR)
    print(f"Fourier. Pérdida final: {loss_f[-1]:.6f}")

    # Entrenamiento con inicialización Aleatoria
    print("[Inicialización Aleatoria]")
    print(f"  Entrenando {EPOCHS} épocas...")
    W1_r, W2_r, loss_r = train_network(x, 'random', EPOCHS, LR)
    print(f"Aleatorio. Pérdida final: {loss_r[-1]:.6f}")

    # Reconstrucciones
    x_hat_f = np.array(forward(W1_f, W2_f, x))
    x_hat_r = np.array(forward(W1_r, W2_r, x))

    # Configuración de subplots según se pida reconstrucción
    if mostrar_reconstruccion:
        fig = make_subplots(rows=2, cols=2,
                            subplot_titles=('Evolución de pérdida',
                                           'Comparación de pérdida final',
                                           'Reconstrucción',
                                           'Error de reconstrucción'))
    else:
        fig = make_subplots(rows=1, cols=2,
                            subplot_titles=('Evolución de pérdida',
                                           'Comparación de pérdida final'))

    # Gráfica 1: Evolución de pérdida (escala logarítmica para mejor visualización)
    fig.add_trace(go.Scatter(y=loss_f, mode='lines', name='Fourier',
                            line=dict(color='#2E86AB', width=2)), row=1, col=1)
    fig.add_trace(go.Scatter(y=loss_r, mode='lines', name='Aleatoria',
                            line=dict(color='#A23B72', width=2)), row=1, col=1)
    fig.update_xaxes(title_text='Iteración', type='log', row=1, col=1)
    fig.update_yaxes(title_text='Pérdida (MSE)', type='log', row=1, col=1)

    # Gráfica 2: Pérdida final (barras comparativas)
    fig.add_trace(go.Bar(x=['Fourier', 'Aleatoria'],
                        y=[loss_f[-1], loss_r[-1]],
                        marker_color=['#2E86AB', '#A23B72'],
                        text=[f'{loss_f[-1]:.6f}', f'{loss_r[-1]:.6f}'],
                        textposition='outside'),
                 row=1, col=2)
    fig.update_yaxes(title_text='Pérdida final (MSE)', row=1, col=2)

    # Gráfica 3: Reconstrucción (comparación señal original vs reconstrucciones)
    if mostrar_reconstruccion:
        fig.add_trace(go.Scatter(y=x, mode='lines', name='Original',
                                line=dict(color='black', width=2)), row=2, col=1)
        fig.add_trace(go.Scatter(y=x_hat_f, mode='lines', name='Fourier',
                                line=dict(color='#2E86AB', width=2)), row=2, col=1)
        fig.add_trace(go.Scatter(y=x_hat_r, mode='lines', name='Aleatoria',
                                line=dict(color='#A23B72', width=2)), row=2, col=1)
        fig.update_xaxes(title_text='Tiempo (muestras)', row=2, col=1)
        fig.update_yaxes(title_text='Amplitud', row=2, col=1)

        # Gráfica 4: Error de reconstrucción (señal original - reconstrucción)
        fig.add_trace(go.Scatter(y=x - x_hat_f, mode='lines', name='Error Fourier',
                                line=dict(color='#2E86AB', width=2)), row=2, col=2)
        fig.add_trace(go.Scatter(y=x - x_hat_r, mode='lines', name='Error Aleatoria',
                                line=dict(color='#A23B72', width=2)), row=2, col=2)
        fig.update_xaxes(title_text='Tiempo (muestras)', row=2, col=2)
        fig.update_yaxes(title_text='Error', row=2, col=2)

    fig.update_layout(height=800, width=1200, showlegend=True)
    fig.show()

    # Resumen estadístico
    print(f"Estadísticas:")
    print(f"Fourier: {loss_f[-1]:.6f}")
    print(f"Aleatoria: {loss_r[-1]:.6f}")
    print(f"Mejor: {'Fourier' if loss_f[-1] < loss_r[-1] else 'Aleatoria'}")

    return {'loss_f': loss_f, 'loss_r': loss_r,
            'final_loss_f': loss_f[-1], 'final_loss_r': loss_r[-1]}

# %% [markdown]
# ### 6.1 Entrenar con señales de cada régimen

# %%
print("\n" + "="*60)
print("Iniacndo entrenamiento")
print(f"{EPOCHS} épocas por entrenamiento")
print("="*60)

"Entrena cada regimen con una señal representativa y las guarda para comparar"
resultados = {}
for regime in [0, 1, 2]:
    idx = np.where(regimenes == regime)[0][0]
    print(f"\n{'#'*60}")
    print(f"# RÉGIMEN {regime} - Señal {idx}")
    print(f"{'#'*60}")
    resultados[regime] = entrenar_y_analizar(signals, regimenes, idx)
    print(f"Régimen {regime} COMPLETADO")

print("Entrenamiento completo")

# %% [markdown]
# ### 6.2 Análisis comparativo por régimen

# %%
def comparar_regimenes(resultados):

    regimes = sorted(resultados.keys())
    final_f = [resultados[r]['final_loss_f'] for r in regimes]
    final_r = [resultados[r]['final_loss_r'] for r in regimes]
    ratios = [f/r for f, r in zip(final_f, final_r)]

    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=('Comparación por régimen',
                                       'Ratio de rendimiento'))

    x = [f'Régimen {r}' for r in regimes]

    # Barras comparativas
    fig.add_trace(go.Bar(x=x, y=final_f, name='Fourier',
                        marker_color='#2E86AB'), row=1, col=1)
    fig.add_trace(go.Bar(x=x, y=final_r, name='Aleatoria',
                        marker_color='#A23B72'), row=1, col=1)
    fig.update_yaxes(title_text='Pérdida final (MSE)', row=1, col=1)

    # Ratio
    fig.add_trace(go.Bar(x=x, y=ratios, name='Ratio',
                        marker_color='#F18F01'), row=1, col=2)
    fig.add_hline(y=1, line_dash='dash', line_color='red',
                  annotation_text='Igual rendimiento', row=1, col=2)
    fig.update_yaxes(title_text='Ratio Fourier/Aleatorio', row=1, col=2)

    fig.update_layout(height=500, width=1200, showlegend=True)
    fig.show()

comparar_regimenes(resultados)

# %% [markdown]
# ## 7. Análisis de compresión

# %%
def analisis_compresion(signals, regimenes):
    print("Analizando compresión")
    resultados = []
    for regime in [0, 1, 2]:
        idx_reg = np.where(regimenes == regime)[0]
        for i in idx_reg[:5]:
            x = signals[i]
            W1_f = fourier_matrix(N, K)
            coefs = np.array(W1_f @ x)
            energia_total = np.sum(coefs**2)
            coefs_abs = np.abs(coefs)
            sorted_idx = np.argsort(coefs_abs)[::-1]
            sorted_coefs = coefs[sorted_idx]
            energia_acum = np.cumsum(sorted_coefs**2) / energia_total
            n_coefs_85 = np.searchsorted(energia_acum, 0.90) + 1
            resultados.append({
                'regimen': regime,
                'n_coeficientes_85': n_coefs_85,
                'porcentaje': (n_coefs_85 / len(coefs)) * 100
            })
    return resultados

resultados_compresion = analisis_compresion(signals, regimenes)

def visualizar_compresion(resultados_compresion):
    print("Generando gráficas de compresión...")

    regimes = [0, 1, 2]
    n_coefs = []
    pct = []
    for r in regimes:
        n_coefs.append([res['n_coeficientes_85'] for res in resultados_compresion if res['regimen'] == r])
        pct.append([res['porcentaje'] for res in resultados_compresion if res['regimen'] == r])

    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=('Coeficientes para 90% de energía',
                                       'Porcentaje de coeficientes'))

    # Boxplots manuales (un espacio para los datos y graficos de cada regimen)
    for i, r in enumerate(regimes):
        fig.add_trace(go.Box(y=n_coefs[i], name=f'Régimen {r}',
                            marker_color='#2E86AB'), row=1, col=1)
        fig.add_trace(go.Box(y=pct[i], name=f'Régimen {r}',
                            marker_color='#A23B72'), row=1, col=2)

    fig.add_hline(y=90, line_dash='dash', line_color='red',
                  annotation_text='Límite 90%', row=1, col=2)
    fig.update_yaxes(title_text='Coeficientes', row=1, col=1)
    fig.update_yaxes(title_text='Porcentaje (%)', row=1, col=2)

    fig.update_layout(height=600, width=1200, showlegend=False)
    fig.show()

    print("\n=== Estadísticas de compresión ===")
    for r in regimes:
        n = [res['n_coeficientes_85'] for res in resultados_compresion if res['regimen'] == r]
        p = [res['porcentaje'] for res in resultados_compresion if res['regimen'] == r]
        print(f"\nRégimen {r}:")
        print(f"  Coeficientes: {np.mean(n):.1f} ± {np.std(n):.1f}")
        print(f"  Porcentaje: {np.mean(p):.1f}% ± {np.std(p):.1f}%")

visualizar_compresion(resultados_compresion)

# %% [markdown]
# ## 7.1 Análisis de compresión - Energía al 90%

# %%
def analisis_compresion_detallado(signals, regimenes, idx):
    """Analiza cuántos coeficientes de Fourier retienen el 90% de la energía"""
    x = signals[idx]
    W1_f = fourier_matrix(N, K)
    coefs = np.array(W1_f @ x)

    # Energía total
    energia_total = np.sum(coefs**2)

    # Ordenar coeficientes por magnitud (de mayor a menor)
    idx_sorted = np.argsort(np.abs(coefs))[::-1]
    coefs_sorted = coefs[idx_sorted]

    # Energía acumulada
    energia_acum = np.cumsum(coefs_sorted**2) / energia_total

    # Encontrar cuántos coeficientes necesarios para 90%
    n_85 = np.argmax(energia_acum >= 0.90) + 1

    print(f"\n{'='*60}")
    print(f"Análisis de compresión - Régimen {regimenes[idx]}")
    print(f"{'='*60}")
    print(f"Coeficientes totales: {len(coefs)}")
    print(f"Coeficientes para 90% de energía: {n_85}")
    print(f"Porcentaje del total: {100 * n_85 / len(coefs):.1f}%")
    print(f"Energía retenida: {100 * energia_acum[n_85-1]:.1f}%")

    # Gráfica
    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=(f'Energía acumulada - Régimen {regimenes[idx]}',
                                       f'Coeficientes ordenados (Top 30)'))

    # Energía acumulada
    fig.add_trace(go.Scatter(y=energia_acum, mode='lines', name='Energía acumulada',
                            line=dict(color='#2E86AB', width=2)), row=1, col=1)
    fig.add_hline(y=0.90, line_dash="dash", line_color="red",
                  annotation_text="90%", annotation_position="bottom right")
    fig.add_vline(x=n_85-1, line_dash="dash", line_color="green",
                  annotation_text=f"n={n_85}", annotation_position="top right")
    fig.update_xaxes(title_text='Número de coeficientes', row=1, col=1)
    fig.update_yaxes(title_text='Energía acumulada', row=1, col=1)

    # Coeficientes ordenados (mostrar top 30)
    fig.add_trace(go.Bar(y=coefs_sorted[:30], name='Coeficientes',
                        marker_color='#A23B72'), row=1, col=2)
    fig.update_xaxes(title_text='Coeficiente (ordenado por magnitud)', row=1, col=2)
    fig.update_yaxes(title_text='Magnitud', row=1, col=2)

    fig.update_layout(height=500, width=1000, showlegend=False)
    fig.show()

    return n_85, energia_acum[n_85-1]

# %%
# Analizar una señal de cada régimen
print("\n" + "="*60)
print("ANÁLISIS DE COMPRESIÓN PARA CADA RÉGIMEN")
print("="*60)

resultados_compresion_detallados = {}
for r in [0, 1, 2]:
    # Encontrar un índice de este régimen
    idx = np.where(regimenes == r)[0][0]
    n_85, energia = analisis_compresion_detallado(signals, regimenes, idx)
    resultados_compresion_detallados[r] = {'n_coefs': n_85, 'energia': energia}

# %%
# Resumen comparativo
print("\n" + "="*60)
print("RESUMEN DE COMPRESIÓN POR RÉGIMEN")
print("="*60)
print(f"{'Régimen':<10} {'Coefs para 90%':<20} {'% del total':<15}")
print("-"*60)
for r in [0, 1, 2]:
    n = resultados_compresion_detallados[r]['n_coefs']
    pct = 100 * n / (2*K)
    print(f"{r:<10} {n:<20} {pct:<15.1f}%")

# Gráfica comparativa
fig = go.Figure()
fig.add_trace(go.Bar(
    x=['Régimen 0\n(sin dispersión)', 'Régimen 1\n(intermedia)', 'Régimen 2\n(fuerte)'],
    y=[resultados_compresion_detallados[0]['n_coefs'],
       resultados_compresion_detallados[1]['n_coefs'],
       resultados_compresion_detallados[2]['n_coefs']],
    marker_color=['#2E86AB', '#F18F01', '#A23B72'],
    text=[f"{resultados_compresion_detallados[0]['n_coefs']} coefs",
          f"{resultados_compresion_detallados[1]['n_coefs']} coefs",
          f"{resultados_compresion_detallados[2]['n_coefs']} coefs"],
    textposition='outside'
))
fig.update_layout(
    title='Coeficientes necesarios para retener 90% de energía',
    xaxis_title='Régimen',
    yaxis_title='Número de coeficientes',
    height=500,
    width=700,
    showlegend=False
)
fig.show()

print("Análisis de compresión completado.")
print("Las señales del Régimen 0 (sin dispersión) son las más compresibles,")
print("mientras que las del Régimen 2 (fuerte dispersión) requieren más coeficientes.")

# %% [markdown]
# ## 8. Análisis de velocidad

# %%
def analizar_velocidad(resultados):
    print("Generando gráficas de velocidad")

    fig = make_subplots(rows=1, cols=3,
                        subplot_titles=('Régimen 0', 'Régimen 1', 'Régimen 2'))

    colors = ['#2E86AB', '#A23B72']
    names = ['Fourier', 'Aleatoria']

    for idx, regime in enumerate([0, 1, 2]):
        loss_f = np.array(resultados[regime]['loss_f'])
        loss_r = np.array(resultados[regime]['loss_r'])

        loss_f_norm = (loss_f - loss_f[-1]) / (loss_f[0] - loss_f[-1])
        loss_r_norm = (loss_r - loss_r[-1]) / (loss_r[0] - loss_r[-1])

        fig.add_trace(go.Scatter(y=loss_f_norm, mode='lines',
                                name=f'Fourier {regime}',
                                line=dict(color='#2E86AB', width=2),
                                showlegend=(idx==0)), row=1, col=idx+1)
        fig.add_trace(go.Scatter(y=loss_r_norm, mode='lines',
                                name=f'Aleatoria {regime}',
                                line=dict(color='#A23B72', width=2),
                                showlegend=(idx==0)), row=1, col=idx+1)

        fig.update_xaxes(title_text='Iteración', row=1, col=idx+1)
        fig.update_yaxes(title_text='Pérdida normalizada', row=1, col=idx+1)
        fig.update_yaxes(range=[-0.05, 1.05], row=1, col=idx+1)

    fig.update_layout(height=500, width=1200, title_text='Velocidad de convergencia comparada')
    fig.show()

analizar_velocidad(resultados)

# %% [markdown]
# ## 9. Respuestas

# %%
"""
===============================================================================
RESPUESTAS A LAS PREGUNTAS GUÍA
===============================================================================

1. ¿AYUDA EMPEZAR CON LA BASE FOURIER VS ALEATORIA?
   Si, la iniciacion de fourier ayuda ya que le brinda la red una base o una
   guia de como es la estructura de los datos, permitiendo que empize en una
   base un poco solida mientras que la entra esta mas a la deriva

2. ¿QUÉ PASA EN EL RÉGIMEN 0?
   En el regimen 0 ambas opcones inician con buenos resultado, sin embargo
   fourier es superior en matener y mejorar los resultados mientras que la
   iniciazion aleatoria tarda mas en lograr los mismos resultados

3. ¿CUÁNTA INFORMACIÓN VIVE EN POCOS COEFICIENTES?
   La mayor parte de la informacion vive en pocos coeficientes,el regimen
   que mas coeficientes ocupa es del 2 usando del 53%-67% y el que menos
   usa es el 0 usadon unicamente del 20%-27%, lo que permite afirmar que
   a menor dispersion menos coeficientes requeridos

===============================================================================
"""

Configuracion completada con Plotly
Parametros: N=256, K=25, LR=0.05, EPOCHS=3000
Señales cargadas: (24, 256)
Régimenes: [2 2 0 0 0 0 1 1 2 1 0 1 2 1 0 1 0 2 1 2 0 2 2 1]
  Régimen 0: 8 señales
  Régimen 1: 8 señales
  Régimen 2: 8 señales
La red neuronal se a implementado
/Verificando Fourier...
/Verificando Fourier para señal 0...


Primeros 5 coeficientes: [-0.01907204  1.7558786   0.11810305  2.1274788  -0.01923633]

Iniacndo entrenamiento
3000 épocas por entrenamiento

############################################################
# RÉGIMEN 0 - Señal 2
############################################################

Señal 2, Régimen 0
[Inicialización Fourier]
  Entrenando 3000 épocas...


  Epoch    0: Loss = 0.489044
  Epoch  200: Loss = 0.000957
  Epoch  400: Loss = 0.000001
  Epoch  600: Loss = 0.000000
  Epoch  800: Loss = 0.000000
  Epoch 1000: Loss = 0.000000
  Epoch 1200: Loss = 0.000000
  Epoch 1400: Loss = 0.000000
  Epoch 1600: Loss = 0.000000
  Epoch 1800: Loss = 0.000000
  Epoch 2000: Loss = 0.000000
  Epoch 2200: Loss = 0.000000
  Epoch 2400: Loss = 0.000000


  Epoch 2600: Loss = 0.000000
  Epoch 2800: Loss = 0.000000
Fourier. Pérdida final: 0.000000
[Inicialización Aleatoria]
  Entrenando 3000 épocas...
  Epoch    0: Loss = 0.488913
  Epoch  200: Loss = 0.003269
  Epoch  400: Loss = 0.000002
  Epoch  600: Loss = 0.000000
  Epoch  800: Loss = 0.000000
  Epoch 1000: Loss = 0.000000
  Epoch 1200: Loss = 0.000000


  Epoch 1400: Loss = 0.000000
  Epoch 1600: Loss = 0.000000
  Epoch 1800: Loss = 0.000000
  Epoch 2000: Loss = 0.000000
  Epoch 2200: Loss = 0.000000
  Epoch 2400: Loss = 0.000000
  Epoch 2600: Loss = 0.000000
  Epoch 2800: Loss = 0.000000
Aleatorio. Pérdida final: 0.000000


...


Análisis de compresión - Régimen 1
Coeficientes totales: 50
Coeficientes para 90% de energía: 6
Porcentaje del total: 12.0%
Energía retenida: 93.3%



Análisis de compresión - Régimen 2
Coeficientes totales: 50
Coeficientes para 90% de energía: 4
Porcentaje del total: 8.0%
Energía retenida: 91.6%



RESUMEN DE COMPRESIÓN POR RÉGIMEN
Régimen    Coefs para 90%       % del total    
------------------------------------------------------------
0          4                    8.0            %
1          6                    12.0           %
2          4                    8.0            %


Análisis de compresión completado.
Las señales del Régimen 0 (sin dispersión) son las más compresibles,
mientras que las del Régimen 2 (fuerte dispersión) requieren más coeficientes.
Generando gráficas de velocidad


'\n===============================================================================\nRESPUESTAS A LAS PREGUNTAS GUÍA\n===============================================================================\n\n1. ¿AYUDA EMPEZAR CON LA BASE FOURIER VS ALEATORIA?\n   Si, la iniciacion de fourier ayuda ya que le brinda la red una base o una\n   guia de como es la estructura de los datos, permitiendo que empize en una\n   base un poco solida mientras que la entra esta mas a la deriva\n\n2. ¿QUÉ PASA EN EL RÉGIMEN 0?\n   En el regimen 0 ambas opcones inician con buenos resultado, sin embargo\n   fourier es superior en matener y mejorar los resultados mientras que la\n   iniciazion aleatoria tarda mas en lograr los mismos resultados\n\n3. ¿CUÁNTA INFORMACIÓN VIVE EN POCOS COEFICIENTES?\n   La mayor parte de la informacion vive en pocos coeficientes,el regimen\n   que mas coeficientes ocupa es del 2 usando del 53%-67% y el que menos\n   usa es el 0 usadon unicamente del 20%-27%, lo que permite afirmar 